In [1]:
import numpy as np

In [2]:
def cek_dominan_diagonal(A):
    """ Memeriksa apakah matriks bersifat Strictly Diagonally Dominant (SDD) """
    n = len(A)
    for i in range(n):
        elemen_diagonal = abs(A[i, i])
        jumlah_lainnya = sum(abs(A[i, j]) for j in range(n) if j != i)
        if elemen_diagonal <= jumlah_lainnya:
            return False
    return True

In [3]:
def iterasi_jacobi(A, B, X0, tol=1e-5, max_iter=100):
    """ Menyelesaikan AX = B secara iteratif menggunakan nilai murni iterasi (k) """
    if not cek_dominan_diagonal(A):
        print("[Peringatan]: Matriks tidak dominan secara diagonal. Konvergensi tidak dijamin.")
        
    n = len(B)
    X = X0.copy()
    X_baru = np.zeros(n)
    
    for hitung_iter in range(1, max_iter + 1):
        for i in range(n):
            # Menggunakan X lama untuk seluruh komputasi di sisi kanan
            s = sum(A[i, j] * X[j] for j in range(n) if j != i)
            X_baru[i] = (B[i] - s) / A[i, i]
            
        # Hitung Error Magnitudo Selisih (Norma Tak Hingga)
        error = np.max(np.abs(X_baru - X))
        if error < tol:
            return X_baru, hitung_iter
            
        X = X_baru.copy() # Salin seluruh nilai serempak untuk iterasi berikutnya
        
    return X, max_iter

In [4]:
def iterasi_gauss_seidel(A, B, X0, tol=1e-5, max_iter=100):
    """ Menyelesaikan AX = B dengan memanfaatkan pembaruan data secara langsung """
    if not cek_dominan_diagonal(A):
        print("[Peringatan]: Matriks tidak dominan secara diagonal. Konvergensi tidak dijamin.")
        
    n = len(B)
    X = X0.copy()
    
    for hitung_iter in range(1, max_iter + 1):
        X_lama = X.copy()
        
        for i in range(n):
            # X[j] di loop ini otomatis bernilai paling baru (ter-update langsung)
            s = sum(A[i, j] * X[j] for j in range(n) if j != i)
            X[i] = (B[i] - s) / A[i, i]
            
        error = np.max(np.abs(X - X_lama))
        if error < tol:
            return X, hitung_iter
            
    return X, max_iter

In [5]:
# =====================================================================
# DEMO STUDI KASUS: DISTRIBUSI PANAS PADA PLAT LOGAM (METODE BEDA HINGGA)
# =====================================================================
if __name__ == "__main__":
    # Matriks Karakteristik Pemodelan Termal Pelat (SDD Mutlak)
    A_termal = np.array([[4.0, -1.0, 0.0],
                         [-1.0, 4.0, -1.0],
                         [0.0, -1.0, 4.0]])
    B_suhu_batas = np.array([100.0, 50.0, 100.0]) # Kondisi batas suhu dinding luar (°C)
    
    # Tebakan awal (Initial Guess) mahasiswa: Asumsi suhu awal nol derajat
    X_tebakan = np.array([0.0, 0.0, 0.0])
    Toleransi_Batas = 1e-5

    print("-" * 60)
    print("ANALISIS NUMERIK ITERASI (STUDI KASUS PROFIL TERMAL)")
    print("-" * 60)
    
    sol_jacobi, iter_j = iterasi_jacobi(A_termal, B_suhu_batas, X_tebakan, tol=Toleransi_Batas)
    print(f"1. Metode Jacobi       -> Solusi: {sol_jacobi} | Selesai dalam {iter_j} Iterasi")
    
    sol_gs, iter_gs = iterasi_gauss_seidel(A_termal, B_suhu_batas, X_tebakan, tol=Toleransi_Batas)
    print(f"2. Metode Gauss-Seidel -> Solusi: {sol_gs} | Selesai dalam {iter_gs} Iterasi")
    print(f"\n[Kesimpulan Dosen]: Gauss-Seidel memotong waktu komputasi sebanyak {iter_j - iter_gs} langkah!")

------------------------------------------------------------
ANALISIS NUMERIK ITERASI (STUDI KASUS PROFIL TERMAL)
------------------------------------------------------------
1. Metode Jacobi       -> Solusi: [32.14285523 28.57142687 32.14285523] | Selesai dalam 16 Iterasi
2. Metode Gauss-Seidel -> Solusi: [32.14285597 28.57142799 32.142857  ] | Selesai dalam 9 Iterasi

[Kesimpulan Dosen]: Gauss-Seidel memotong waktu komputasi sebanyak 7 langkah!
